<a href="https://colab.research.google.com/github/Satyam-Mittal2527/PyTorch/blob/main/pytorch_training_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [88]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [89]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [90]:
df.shape

(569, 33)

In [91]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [92]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [93]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [94]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [95]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

## Numpy arrays to PyTorch Tensors

In [96]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

## Defining the model

In [97]:
class MySimpleNN():

  def __init__(self, X):
    self.weights = torch.rand(X.shape[1], 1, dtype = torch.float64, requires_grad = True)
    self.bias = torch.zeros(1, dtype = torch.float64, requires_grad = True)

  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
    loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
    return loss


## important Parameters

In [98]:
learning_rate = 0.1
epochs = 25

## Training Pipeline

In [99]:
model = MySimpleNN(X_train_tensor)
#define loop
for epoch in range(epochs):

  #Forward Pass
  y_pred = model.forward(X_train_tensor)

  #loss calculation
  loss = model.loss_function(y_pred, y_train_tensor)

  #backward pass
  loss.backward()

  #Update parameters
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad

  # Zero gradient
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  #print loss in each epoch
  print(f"Epoch: {epoch +1}, loss : {loss.item()}")

Epoch: 1, loss : 3.9948646231830574
Epoch: 2, loss : 3.8853411511953846
Epoch: 3, loss : 3.774905938588124
Epoch: 4, loss : 3.6611584497258347
Epoch: 5, loss : 3.541901894425506
Epoch: 6, loss : 3.416084042838488
Epoch: 7, loss : 3.2821434873499897
Epoch: 8, loss : 3.1421215404675147
Epoch: 9, loss : 2.996980954922363
Epoch: 10, loss : 2.8508114087702827
Epoch: 11, loss : 2.7020366411896117
Epoch: 12, loss : 2.543127818147527
Epoch: 13, loss : 2.3756375316663614
Epoch: 14, loss : 2.208199603461888
Epoch: 15, loss : 2.0409714139616035
Epoch: 16, loss : 1.8722961833700267
Epoch: 17, loss : 1.7077804441459892
Epoch: 18, loss : 1.5529287941979715
Epoch: 19, loss : 1.4092297627850456
Epoch: 20, loss : 1.2755152518476496
Epoch: 21, loss : 1.158260039712666
Epoch: 22, loss : 1.0590532161882367
Epoch: 23, loss : 0.9785612392495867
Epoch: 24, loss : 0.9162406024032068
Epoch: 25, loss : 0.8702515148174023


## Model Evaluation

In [100]:
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5549399852752686


## NN Module implementation

In [101]:
import torch.nn as nn
class MySimpleNN(nn.Module):

  def __init__(self, num_features):

    super().__init__()

    self.network = nn.Sequential(
        nn.Linear(num_features,1),
        nn.Sigmoid()
    )
  def forward(self, X):
    y_pred = self.network(X)

    return y_pred


In [102]:
loss_function = nn.BCELoss()

In [104]:
model2 = MySimpleNN(X_train_tensor.shape[1])

#define optimizer
optimizer = torch.optim.SGD(model2.parameters(), lr = learning_rate)

X_train_tensor = X_train_tensor.float()
y_train_tensor = y_train_tensor.float()
X_test_tensor = X_test_tensor.float()
for epoch in range(epochs):

  y_pred = model2(X_train_tensor)

  #loss calculation
  loss = loss_function(y_pred, y_train_tensor.reshape(-1,1))

  #Clear gradients
  optimizer.zero_grad()

  #backward
  loss.backward()

  #Update parameters
  optimizer.step()


  #print loss in each epoch
  print(f"Epoch: {epoch +1}, loss : {loss.item()}")



Epoch: 1, loss : 0.8579093217849731
Epoch: 2, loss : 0.6058536171913147
Epoch: 3, loss : 0.47989919781684875
Epoch: 4, loss : 0.40963688492774963
Epoch: 5, loss : 0.3642584979534149
Epoch: 6, loss : 0.3320440351963043
Epoch: 7, loss : 0.3077130615711212
Epoch: 8, loss : 0.28851792216300964
Epoch: 9, loss : 0.27287784218788147
Epoch: 10, loss : 0.2598142921924591
Epoch: 11, loss : 0.2486867606639862
Epoch: 12, loss : 0.23905682563781738
Epoch: 13, loss : 0.23061367869377136
Epoch: 14, loss : 0.2231300324201584
Epoch: 15, loss : 0.2164354920387268
Epoch: 16, loss : 0.2103995382785797
Epoch: 17, loss : 0.20492023229599
Epoch: 18, loss : 0.19991667568683624
Epoch: 19, loss : 0.19532369077205658
Epoch: 20, loss : 0.19108811020851135
Epoch: 21, loss : 0.1871659904718399
Epoch: 22, loss : 0.18352070450782776
Epoch: 23, loss : 0.18012145161628723
Epoch: 24, loss : 0.17694203555583954
Epoch: 25, loss : 0.17396004498004913


In [105]:
with torch.no_grad():
  y_pred = model2.forward(X_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5226223468780518
